# Stage 0 Kaggle Training

Ready-to-run Kaggle notebook for Stage 0 training with automatic Hugging Face Hub checkpoint upload and resume.

Setup checklist: Accelerator = GPU T4 x2, Internet = ON, add a Kaggle secret named `HF_TOKEN` containing a Hugging Face WRITE token.


In [ ]:
VARIANT = "pdr"  # "pdr" | "gla" | "transformer"
HF_USERNAME = "your-username"
REPO_URL = "https://github.com/your-username/OMNI.git"  # or leave and upload repo as Kaggle dataset
TOKENS = 2_500_000_000
MAX_HOURS = 8.5

HUB_REPO = f"{HF_USERNAME}/stage0-{VARIANT}"
print(f"variant={VARIANT} hub_repo={HUB_REPO}")


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "OMNI"
TRAIN_DIR = REPO_DIR / "train"

def run(cmd, *, cwd=None):
    print("+ " + " ".join(str(part) for part in cmd), flush=True)
    subprocess.run(cmd, cwd=cwd, check=True)

repo_url = REPO_URL.strip()
placeholder_repo = (not repo_url) or "your-username" in repo_url
if REPO_DIR.exists():
    if (REPO_DIR / ".git").exists() and not placeholder_repo:
        run(["git", "pull", "--ff-only"], cwd=REPO_DIR)
    else:
        print(f"Using existing repo at {REPO_DIR}", flush=True)
else:
    if placeholder_repo:
        candidates = sorted(Path("/kaggle/input").glob("**/train/run_stage0.py"))
        if not candidates:
            raise RuntimeError("Set REPO_URL to your OMNI GitHub repo, or attach an OMNI Kaggle dataset containing train/run_stage0.py.")
        dataset_root = candidates[0].parents[1]
        shutil.copytree(
            dataset_root,
            REPO_DIR,
            ignore=shutil.ignore_patterns(".git", "__pycache__", ".pytest_cache", ".mypy_cache", "runs", ".pytest_tmp"),
        )
        print(f"Copied OMNI dataset from {dataset_root} to {REPO_DIR}", flush=True)
    else:
        run(["git", "clone", repo_url, str(REPO_DIR)])

# Install only non-torch deps: Kaggle preinstalls a CUDA-matched torch that
# a requirements upgrade would replace with a multi-GB, possibly broken wheel.
reqs = [
    line.strip()
    for line in (TRAIN_DIR / "requirements.txt").read_text().splitlines()
    if line.strip() and not line.strip().startswith(("#", "torch"))
]
run([sys.executable, "-m", "pip", "install", "-q", *reqs])


In [ ]:
import os

from kaggle_secrets import UserSecretsClient
from huggingface_hub import create_repo

if VARIANT not in {"pdr", "gla", "transformer"}:
    raise ValueError('VARIANT must be one of: "pdr", "gla", "transformer"')
if not HF_USERNAME or HF_USERNAME == "your-username":
    raise ValueError("Set HF_USERNAME in the config cell before running training.")

try:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    raise RuntimeError("Missing Kaggle secret HF_TOKEN. Add a secret named HF_TOKEN containing a Hugging Face WRITE token.") from exc

if not hf_token:
    raise RuntimeError("Kaggle secret HF_TOKEN is empty. Store a Hugging Face WRITE token in that secret.")

os.environ["HF_TOKEN"] = hf_token
create_repo(HUB_REPO, exist_ok=True, private=True, token=hf_token)
print(f"HF_TOKEN loaded and Hub repo ready: {HUB_REPO}")


In [ ]:
run([sys.executable, "-m", "pytest", "train/tests", "-q"], cwd=REPO_DIR)
run([sys.executable, "-c", "from perspective_torch import param_table; param_table()"], cwd=TRAIN_DIR)


In [ ]:
output_dir = f"/kaggle/working/stage0-{VARIANT}"
run(
    [
        sys.executable,
        "train/run_stage0.py",
        "--variant",
        VARIANT,
        "--tokens",
        str(TOKENS),
        "--hub-repo",
        HUB_REPO,
        "--max-hours",
        str(MAX_HOURS),
        "--output-dir",
        output_dir,
    ],
    cwd=REPO_DIR,
)


## Next session

Run All again with the same `VARIANT`; resume is automatic from the Hub repo when no local checkpoint exists.

Watch metrics in `checkpoints/metrics.jsonl` inside the Hub repo for that variant.
